In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, Dataset
import sys 
sys.path.append("../")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 10

# Load the SVHN dataset
train_ds = torchvision.datasets.SVHN(root='./data', split='train', download=True)
test_ds = torchvision.datasets.SVHN(root='./data', split='test', download=True)

print(f"Training samples: {len(train_ds)}, Test samples: {len(test_ds)}")

mean = torch.tensor([0.4377, 0.4438, 0.4728])
std = torch.tensor([0.1980, 0.2010, 0.1970])

Training samples: 73257, Test samples: 26032


In [2]:
class RandomRangAugment(transforms.RandAugment):
    def __init__(self, num_ops=2):
        super().__init__(num_ops=num_ops)

    def __call__(self, img):
        # just randaugment with random magnitude
        self.magnitude = torch.randint(0, self.num_magnitude_bins, (1,)).item()
        img = super().__call__(img)
        return img

In [3]:
norm_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

weak_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

strong_transform=transforms.Compose([
        RandomRangAugment(num_ops=2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
        transforms.RandomErasing(),
    ])

In [4]:
class UnlabeledDataset(Dataset):
    def __init__(self, dataset, weak_transform, strong_transform):
        self.dataset = dataset
        self.weak_transform = weak_transform
        self.strong_transform = strong_transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, _ = self.dataset[idx]
        x_w = self.weak_transform(x)
        x_s = self.strong_transform(x)
        return x_w, x_s, idx

In [ ]:
from datasets import TransformedDataset
from wideresnet2 import WideResNet 
from utils import evaluate_f1_and_accuracy
import os
import json
import time
import numpy as np

num_runs = 3
alpha = 0.75
T = 0.5

test_ds = TransformedDataset(test_ds, norm_transform)

for run in range(num_runs):
    print(f"Run {run+1}/{num_runs}")

    # Init random seeds for reproducibility
    torch.manual_seed(run)
    num_labeled = 40
    perm_indices = torch.randperm(len(train_ds))
    labeled_indices = perm_indices[:num_labeled]
    unlabeled_indices = perm_indices[num_labeled:]

    # Create datasets and dataloaders
    labeled_ds = TransformedDataset(Subset(train_ds, labeled_indices), weak_transform)
    unlabeled_ds = UnlabeledDataset(Subset(train_ds, unlabeled_indices), weak_transform, weak_transform)

    # Create dataloaders
    batch_size = 64
    mu = 1
    labeled_loader = DataLoader(labeled_ds, batch_size=batch_size, shuffle=True)
    unlabeled_loader = DataLoader(unlabeled_ds, batch_size=batch_size*mu, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    # Create iterators for the dataloaders
    labeled_iter = iter(labeled_loader)
    unlabeled_iter = iter(unlabeled_loader)

    # Define the model
    model = WideResNet(depth=28, widen_factor=2, num_classes=num_classes).to(device)
    max_steps = 15_270  # Equivalent to 100 epochs on the full dataset with batch size 64
    optimizer = torch.optim.SGD(model.parameters(), lr=0.03, momentum=0.9, weight_decay=5e-4, nesterov=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_steps)

    # Settings
    method_name = "MixMatch"
    name_of_experiment = f"svhn_{num_labeled}_labels_run_{run+1}"

    if os.path.exists(f"results/{name_of_experiment}/{method_name}.json"):
        print(f"Results for {name_of_experiment} already exist. Skipping training to avoid overwriting.")
        continue # this will skip the rest of the training loop and move to the next run

    # Metrics to track
    metrics = {
    "test_f1": [0.0],  # Start with 0% F1 before training
    "test_acc": [0.0],  # Start with 0% accuracy before training
    "budget": [0]
    }

    # Hyperparameters
    alpha = 0.75
    T = 0.5
    budget_per_iteration = 1 + 2 * mu  # 1 batch of labeled + 2 batches of unlabeled (weak + weak)
    test_budget_period = 900  # Evaluate on test set every 900 batches seen

    # Training loop
    current_budget = 0
    start_time = time.time()
    for step in range(max_steps):
        running_loss = 0.0
        running_loss_sup = 0.0
        running_loss_unsup = 0.0
        model.train()
        try:
            x_l, y_l = next(labeled_iter)
        except StopIteration:
            labeled_iter = iter(labeled_loader)
            x_l, y_l = next(labeled_iter)

        try:
            x_u_1, x_u_2, _ = next(unlabeled_iter)
        except StopIteration:
            unlabeled_iter = iter(unlabeled_loader)
            x_u_1, x_u_2, _ = next(unlabeled_iter)

        x_l, y_l = x_l.to(device), y_l.to(device)
        x_u_1, x_u_2 = x_u_1.to(device), x_u_2.to(device)

        # unsupervised
        with torch.no_grad():
            logits_u_1 = model(x_u_1)
            logits_u_2 = model(x_u_2)
            probs_u_1 = F.softmax(logits_u_1, dim=1)
            probs_u_2 = F.softmax(logits_u_2, dim=1)
            probs_avg = (probs_u_1 + probs_u_2) / 2
            
            # Sharpening
            probs_sharpened = probs_avg ** (1 / T)
            probs_sharpened = probs_sharpened / probs_sharpened.sum(dim=1, keepdim=True)

        # Concatenate labeled and unlabeled data
        all_inputs = torch.cat([x_l, x_u_1, x_u_2], dim=0)
        all_targets = torch.cat([F.one_hot(y_l, num_classes), probs_sharpened, probs_sharpened], dim=0)

        # Shuffle the combined data
        indices = torch.randperm(all_inputs.size(0))
        all_inputs = all_inputs[indices]
        all_targets = all_targets[indices]

        # Mixup
        lam_x = torch.tensor([np.random.beta(alpha, alpha) for _ in range(x_l.size(0))], dtype=torch.float32, device=device).unsqueeze(1).unsqueeze(2).unsqueeze(3)
        lam_u = torch.tensor([np.random.beta(alpha, alpha) for _ in range(x_u_1.size(0) + x_u_2.size(0))], dtype=torch.float32, device=device).unsqueeze(1).unsqueeze(2).unsqueeze(3)

        mixup_x = lam_x.view(-1, 1, 1, 1) * x_l + (1 - lam_x.view(-1, 1, 1, 1)) * all_inputs[:x_l.size(0)]
        mixup_u = lam_u.view(-1, 1, 1, 1) * torch.cat([x_u_1, x_u_2], dim=0) + (1 - lam_u.view(-1, 1, 1, 1)) * all_inputs[x_l.size(0):]

        mixup_targets_x = lam_x.view(-1, 1) * F.one_hot(y_l, num_classes) + (1 - lam_x.view(-1, 1)) * all_targets[:x_l.size(0)]
        mixup_targets_u = lam_u.view(-1, 1) * torch.cat([probs_sharpened, probs_sharpened], dim=0) + (1 - lam_u.view(-1, 1)) * all_targets[x_l.size(0):]

        # Forward pass
        logits_mixup_x = model(mixup_x)
        logits_mixup_u = model(mixup_u)

        # Compute losses
        loss_x = F.cross_entropy(logits_mixup_x, mixup_targets_x.argmax(dim=1))

        #Sharpen logits before computing MSE loss
        logits_mixup_u_sharpened = logits_mixup_u ** (1 / T)
        logits_mixup_u_sharpened = logits_mixup_u_sharpened / logits_mixup_u_sharpened.sum(dim=1, keepdim=True)

        loss_u = F.mse_loss(logits_mixup_u_sharpened, mixup_targets_u)
        loss= loss_x + (min(step, 16000)/16000) * 100 * loss_u

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        running_loss_sup += loss_x.item()
        running_loss_unsup += loss_u.item()

        current_budget += budget_per_iteration

        if current_budget % test_budget_period < budget_per_iteration:
            f1, accuracy = evaluate_f1_and_accuracy(model, test_loader, device)
            metrics["test_f1"].append(f1)
            metrics["test_acc"].append(accuracy)
            metrics["budget"].append(current_budget)
            elapsed_time = time.time() - start_time
            print(f"Step {step+1}/{max_steps}, Budget: {current_budget}, Loss: {running_loss:.4f}, Sup Loss: {running_loss_sup:.4f}, Unsup Loss: {running_loss_unsup:.4f}, Test F1: {f1:.4f}, Test Acc: {accuracy:.4f}, Elapsed Time: {elapsed_time:.2f}s", flush=True)

            os.makedirs("results", exist_ok=True)
            os.makedirs(f"results/{name_of_experiment}", exist_ok=True)
            with open(f"results/{name_of_experiment}/{method_name}.json", "w") as f:
                json.dump(metrics, f, indent=4)

            # Early stopping if divergence is detected (accuracy does not go above 11% for 5 consecutive evaluations)
            early_stopping = (num_labeled == 40) and len(metrics["test_acc"]) > 5 and all(acc <= 0.11 for acc in metrics["test_acc"][-5:])
            if early_stopping:
                print(f"Early stopping at step {step+1} due to accuracy stagnating.")
                
                # Fill the remaining metrics with the last known values until max_steps
                for remaining_step in range(step+1, max_steps):
                    current_budget += budget_per_iteration
                    if current_budget % test_budget_period < budget_per_iteration:
                        metrics["test_f1"].append(metrics["test_f1"][-1])  # Append last known F1
                        metrics["test_acc"].append(metrics["test_acc"][-1])  # Append last known accuracy
                        metrics["budget"].append(current_budget)
                # Save the final metrics after early stopping
                with open(f"results/{name_of_experiment}/{method_name}.json", "w") as f:
                    json.dump(metrics, f, indent=4)
                break  

        elif step+1 == max_steps: # Final evaluation at the end of training
            f1, accuracy = evaluate_f1_and_accuracy(model, test_loader, device)
            metrics["test_f1"].append(f1)
            metrics["test_acc"].append(accuracy)
            metrics["budget"].append(current_budget)
            elapsed_time = time.time() - start_time
            print(f"Step {step+1}/{max_steps}, Budget: {current_budget}, Loss: {running_loss:.4f}, Sup Loss: {running_loss_sup:.4f}, Unsup Loss: {running_loss_unsup:.4f}, Test F1: {f1:.4f}, Test Acc: {accuracy:.4f}, Elapsed Time: {elapsed_time:.2f}s", flush=True)

            os.makedirs("results", exist_ok=True)
            os.makedirs(f"results/{name_of_experiment}", exist_ok=True)
            with open(f"results/{name_of_experiment}/{method_name}.json", "w") as f:
                json.dump(metrics, f, indent=4) 

        else:
            print(f"Step {step+1}/{max_steps}, Budget: {current_budget}, Loss: {running_loss:.4f}, Sup Loss: {running_loss_sup:.4f}, Unsup Loss: {running_loss_unsup:.4f}", end="\r", flush=True)

Run 1/3
Step 300/15270, Budget: 900, Loss: 1.1654, Sup Loss: 1.1301, Unsup Loss: 0.0189, Test F1: 0.1502, Test Acc: 0.1755, Elapsed Time: 43.47s
Step 600/15270, Budget: 1800, Loss: 0.6388, Sup Loss: 0.5734, Unsup Loss: 0.0175, Test F1: 0.1586, Test Acc: 0.1907, Elapsed Time: 85.02s
Step 900/15270, Budget: 2700, Loss: 0.7483, Sup Loss: 0.6690, Unsup Loss: 0.0141, Test F1: 0.1644, Test Acc: 0.1918, Elapsed Time: 126.86s
Step 1200/15270, Budget: 3600, Loss: 0.6141, Sup Loss: 0.5143, Unsup Loss: 0.0133, Test F1: 0.1840, Test Acc: 0.2175, Elapsed Time: 168.20s
Step 1500/15270, Budget: 4500, Loss: 0.4742, Sup Loss: 0.3375, Unsup Loss: 0.0146, Test F1: 0.1593, Test Acc: 0.1995, Elapsed Time: 209.52s
Step 1800/15270, Budget: 5400, Loss: 1.0298, Sup Loss: 0.8200, Unsup Loss: 0.0187, Test F1: 0.1861, Test Acc: 0.2263, Elapsed Time: 251.22s
Step 2100/15270, Budget: 6300, Loss: 0.6373, Sup Loss: 0.4528, Unsup Loss: 0.0141, Test F1: 0.1979, Test Acc: 0.2387, Elapsed Time: 292.82s
Step 2400/15270, B